Here, we try to use the curated datasets to insert them into our MySQL database and test the data. First start the mysql server

In [22]:
%pip install mysql-connector-python


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [23]:
import sys
print(sys.executable)

import mysql.connector
print("Import funktioniert")

/usr/local/bin/python3
Import funktioniert


In [24]:
import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="newpassword",
    database="hurricanes"
)

cursor = connection.cursor()

print("Connected to:", connection.database)
print(type(connection))
print(type(cursor))
'''
I have mysql server on device called "hurricanes"
what is the mysql.connector?
mysql.connector is a Python library that allows you to connect to a MySQL database server from your Python code. 
It provides methods to execute SQL queries, retrieve data, and manage database connections. 
In this code, it is used to connect to the MySQL server running on the device called "hurricanes" and interact with the "hurricanes" database.


'''

Connected to: hurricanes
<class 'mysql.connector.connection_cext.CMySQLConnection'>
<class 'mysql.connector.cursor_cext.CMySQLCursor'>


'\nI have mysql server on device called "hurricanes"\nwhat is the mysql.connector?\nmysql.connector is a Python library that allows you to connect to a MySQL database server from your Python code. \nIt provides methods to execute SQL queries, retrieve data, and manage database connections. \nIn this code, it is used to connect to the MySQL server running on the device called "hurricanes" and interact with the "hurricanes" database.\n\n\n'

In [25]:
print(type(connection))
print(type(cursor))

<class 'mysql.connector.connection_cext.CMySQLConnection'>
<class 'mysql.connector.cursor_cext.CMySQLCursor'>


delete every row in all relations before adding actual data


In [33]:
def removeAllDestructionData():
    query = """
    DELETE FROM Destruction
    """
    cursor.execute(query)
    connection.commit()

def removeAllMeteorologicalData():
    query = """
    DELETE FROM MeteorologicalData
    """
    cursor.execute(query)
    connection.commit()

def removeAllLocationData():
    query = """
    DELETE FROM Location
    """
    cursor.execute(query)
    connection.commit()

def removeAllHurricaneData():
    query = """
    DELETE FROM Hurricane
    """
    cursor.execute(query)
    connection.commit()









In [38]:
removeAllDestructionData()
removeAllMeteorologicalData()
removeAllLocationData()
removeAllHurricaneData()

for t in ["Destruction", "MeteorologicalData", "Location", "Hurricane"]:
    cursor.execute(f"SELECT COUNT(*) FROM {t}")
    print(t, cursor.fetchone())

Destruction (0,)
MeteorologicalData (0,)
Location (0,)
Hurricane (0,)


change constraints of tables according to the new ERD


In [35]:
def alterDestruction():
    query = """
    ALTER TABLE Destruction
        MODIFY destruction_id INT NOT NULL,
        MODIFY deaths_direct INT NULL,
        MODIFY deaths_indirect INT NULL,
        MODIFY infrastructure_damage_usd BIGINT NULL,
        MODIFY economic_damage_overall_usd BIGINT NULL,
        ADD COLUMN source TEXT NOT NULL;
    """
    cursor.execute(query)
    connection.commit()

def alterLocation():
    query = """
    ALTER TABLE Location
        MODIFY location_id INT NOT NULL,
        MODIFY latitude DECIMAL(9,6) NULL,
        MODIFY longitude DECIMAL(9,6) NULL,
        MODIFY region VArCHAR(120) NULL,
        MODIFY country VARCHAR(120) NULL,
        ADD COLUMN source TEXT NOT NULL;
    """
    cursor.execute(query)
    connection.commit()

def alterMeteorologicalData():
    query = """
    ALTER TABLE MeteorologicalData
        MODIFY md_id INT NOT NULL,
        MODIFY avg_wind_speed DECIMAL(6,2) NULL,
        MODIFY total_rainfall DECIMAL(7,2) NULL,
        MODIFY avg_temperature DECIMAL(5,2) NULL,
        ADD COLUMN source TEXT NOT NULL;
    """
    cursor.execute(query)
    connection.commit()

def alterHurricane():
    query = """
    ALTER TABLE Hurricane
        MODIFY hurricane_id INT NOT NULL,
        ADD COLUMN source TEXT NOT NULL;
    """
    cursor.execute(query)
    connection.commit()

add real data from Databases-week5-task.xlsx

In [31]:
%pip install openpyxl


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [42]:
import pandas as pd

In [45]:
FILE = "/Volumes/My-Disk/Uni/databases/week3/createSchema/Databases-Group17-hurricanesDB/week5/Databases-week5-task.xlsx"
d = pd.read_excel(FILE, sheet_name="destruction_entity", header=4)
m = pd.read_excel(FILE, sheet_name="meteorological_data", header=4)
bad = d[d["MD_ID"].notna() & ~d["MD_ID"].isin(m["MD_ID"])]
print(bad[["destruction_ID", "MD_ID"]].assign(excel_row=bad.index + 6))

    destruction_ID  MD_ID  excel_row
28              28    123         34


In [ ]:
import getpass

import mysql.connector
import pandas as pd

FILE = "week5/Databases-week5-task.xlsx"
SHEETS = [("hurricane_entity", "Hurricane"), ("location_entity", "Location"),
          ("meteorological_data", "MeteorologicalData"), ("destruction_entity", "Destruction")]

conn = mysql.connector.connect(host="localhost", user="root", database="hurricanes",
                               password=getpass.getpass("MySQL password: "))
cur = conn.cursor()
# keep ID 0 as 0 even if an ID column is AUTO_INCREMENT
cur.execute("SET SESSION sql_mode = CONCAT_WS(',', @@sql_mode, 'NO_AUTO_VALUE_ON_ZERO')")


def clean(col, v):
    if pd.isna(v):
        return None                                   # empty cell -> NULL
    if isinstance(v, pd.Timestamp):
        return v.to_pydatetime()
    if col.endswith("_datetime") and isinstance(v, str):
        return pd.to_datetime(v.replace("UTC", "").strip()).to_pydatetime()
    if col.endswith("_usd") and isinstance(v, str):
        return int(v.replace("$", "").replace(".", ""))   # '$2.500.000.000'
    if hasattr(v, "item"):
        v = v.item()
    return int(v) if isinstance(v, float) and v.is_integer() else v


try:
    # stop early if a table already has rows (this is what causes "Duplicate entry")
    for _, table in SHEETS:
        cur.execute(f"SELECT COUNT(*) FROM {table}")
        n = int(str((cur.fetchone() or (0,))[0]))  # type: ignore
        if n:
            raise RuntimeError(f"{table} already has {n} rows. Empty the tables first (Destruction, Meteorological_Data, Location, Hurricane).")

    for sheet, table in SHEETS:
        df = pd.read_excel(FILE, sheet_name=sheet, header=4).dropna(how="all")  # headers on row 5
        df = df[df.iloc[:, 0].notna()]                                          # drop stray text rows
        cur.execute(f"SHOW COLUMNS FROM {table}")
        db_cols = {str(r[0]).lower(): str(r[0]) for r in cur.fetchall()}  # type: ignore
        cols = [c for c in map(str, df.columns) if c.lower() in db_cols]        # skips 'source'
        df = df[cols]
        sql = (f"INSERT INTO {table} ({', '.join(db_cols[c.lower()] for c in cols)}) "
               f"VALUES ({', '.join(['%s'] * len(cols))})")
        rows = [tuple(clean(c, v) for c, v in zip(cols, r)) for r in df.itertuples(index=False)]
        cur.executemany(sql, rows)
        print(table, len(rows), "rows")

    conn.commit()
    print("Done.")
except Exception:
    conn.rollback()   # undo everything so a failed run leaves no locks or half-loaded tables
    raise
finally:
    cur.close()
    conn.close()

Hurricane 3 rows
Location 99 rows
MeteorologicalData 109 rows
Destruction 29 rows
Done.


In [ ]:
def get_specific_hurricanes(start_datetime, end_datetime):
    query = """
    SELECT * FROM Hurricane
    WHERE start_datetime >= %s AND end_datetime <= %s
    """
    cursor.execute(query, (start_datetime, end_datetime))
    return cursor.fetchall()

def get_meteorological_data_on_hurricane(hurricane_id):
    query = """
    SELECT * FROM MeteorologicalData
    WHERE hurricane_id = %s
    """
    cursor.execute(query, (hurricane_id,))
    return cursor.fetchall()

def get_sum_damage(location):
    sql = """
        SELECT Location.region,
               SUM(Destruction.deaths_direct),
               SUM(Destruction.deaths_indirect),
               SUM(Destruction.infrastructure_damage_usd),
               SUM(Destruction.economic_damage_overall_usd)
        FROM Location
        JOIN MeteorologicalData ON Location.location_id = MeteorologicalData.location_id
        JOIN Destruction ON MeteorologicalData.md_id = Destruction.md_id
        WHERE Location.region = %s
        GROUP BY Location.region;
    """
    cursor.execute(sql, (location,))
    return cursor.fetchone()

In [1]:
for row in get_specific_hurricanes("2022-01-01", "2022-12-31"):
    print(row)

for row in get_meteorological_data_on_hurricane(1):
    print(row)

print(get_sum_damage("St. Elizabeth"))

NameError: name 'get_specific_hurricanes' is not defined